In [0]:
# ── Load credentials from config notebook ────────────────────────────────────
%run ./config

# ── Read Gold parquet from S3 ─────────────────────────────────────────────────
df_customers = spark.read \
    .option("fs.s3a.access.key", ACCESS_KEY) \
    .option("fs.s3a.secret.key", SECRET_KEY) \
    .option("fs.s3a.session.token", SESSION_TOKEN) \
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider") \
    .parquet(f"{S3_GOLD_BASE}/customer_analytics/")
df_customers.createOrReplaceTempView("customer_analytics")

print(f"✅ {df_customers.count()} rows loaded")

Warning you are using the ipython `%run` line magic. To use the databricks `%run` cell magic make sure that the magic is at the very start of the cell.

✅ Credentials loaded (use as .option() in read/write calls)
✅ 10000 rows loaded


In [0]:
%pip install plotly pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [0]:
df = spark.sql("""
    SELECT 
        customer_segment,
        COUNT(customer_id) AS total_customers
    FROM customer_analytics
    GROUP BY customer_segment
    ORDER BY total_customers DESC
""").toPandas()

fig = px.pie(
    df,
    names="customer_segment",
    values="total_customers",
    title="👥 Customer Distribution by Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4
)
fig.update_traces(textinfo="percent+label")
fig.show()

In [0]:
df = spark.sql("""
    SELECT 
        country,
        COUNT(customer_id) AS total_customers
    FROM customer_analytics
    GROUP BY country
    ORDER BY total_customers DESC
""").toPandas()

fig = px.bar(
    df,
    x="country",
    y="total_customers",
    title="🌍 Customer Count by Country",
    color="country",
    color_discrete_sequence=px.colors.qualitative.Pastel,
    text="total_customers"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Country",
    yaxis_title="Customers"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT 
        customer_segment,
        ROUND(AVG(lifetime_value), 2)  AS avg_lifetime_value,
        ROUND(AVG(avg_order_value), 2) AS avg_order_value,
        ROUND(AVG(total_orders), 2)    AS avg_orders
    FROM customer_analytics
    GROUP BY customer_segment
    ORDER BY avg_lifetime_value DESC
""").toPandas()

fig = px.bar(
    df,
    x="customer_segment",
    y="avg_lifetime_value",
    title="💰 Average Lifetime Value by Customer Segment",
    color="customer_segment",
    color_discrete_sequence=px.colors.qualitative.Set1,
    text="avg_lifetime_value"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Segment",
    yaxis_title="Avg Lifetime Value ($)"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT 
        customer_segment,
        ROUND(AVG(total_orders), 1)    AS avg_orders,
        ROUND(AVG(avg_order_value), 2) AS avg_order_value
    FROM customer_analytics
    GROUP BY customer_segment
""").toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Avg Orders per Customer", "Avg Order Value ($)")
)

fig.add_trace(
    go.Bar(
        x=df["customer_segment"],
        y=df["avg_orders"],
        name="Avg Orders",
        marker_color="#636EFA",
        text=df["avg_orders"],
        textposition="outside"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=df["customer_segment"],
        y=df["avg_order_value"],
        name="Avg Order Value",
        marker_color="#EF553B",
        text=df["avg_order_value"],
        textposition="outside"
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="📦 Order Behaviour by Segment",
    showlegend=False
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT 
        preferred_payment_method,
        COUNT(customer_id) AS total_customers
    FROM customer_analytics
    WHERE preferred_payment_method IS NOT NULL
    GROUP BY preferred_payment_method
    ORDER BY total_customers DESC
""").toPandas()

fig = px.bar(
    df,
    x="preferred_payment_method",
    y="total_customers",
    title="💳 Preferred Payment Method",
    color="preferred_payment_method",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    text="total_customers"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Payment Method",
    yaxis_title="Customers"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT 
        customer_segment,
        ROUND(AVG(loyalty_points), 0) AS avg_loyalty_points
    FROM customer_analytics
    GROUP BY customer_segment
    ORDER BY avg_loyalty_points DESC
""").toPandas()

fig = px.bar(
    df,
    x="customer_segment",
    y="avg_loyalty_points",
    title="⭐ Average Loyalty Points by Segment",
    color="customer_segment",
    color_discrete_sequence=px.colors.qualitative.Safe,
    text="avg_loyalty_points"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Segment",
    yaxis_title="Avg Loyalty Points"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT 
        customer_segment,
        annual_income
    FROM customer_analytics
    WHERE annual_income IS NOT NULL
""").toPandas()

fig = px.box(
    df,
    x="customer_segment",
    y="annual_income",
    title="💵 Annual Income Distribution by Segment",
    color="customer_segment",
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.update_layout(
    showlegend=False,
    xaxis_title="Segment",
    yaxis_title="Annual Income ($)"
)
fig.show()

In [0]:
df = spark.sql("""
    SELECT 
        customer_status,
        COUNT(customer_id) AS total_customers
    FROM customer_analytics
    GROUP BY customer_status
    ORDER BY total_customers DESC
""").toPandas()

fig = px.pie(
    df,
    names="customer_status",
    values="total_customers",
    title="🔵 Active vs Inactive Customers",
    color_discrete_sequence=px.colors.qualitative.Set3,
    hole=0.35
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [0]:
df = spark.sql("""
    SELECT
        COUNT(customer_id)             AS total_customers,
        ROUND(AVG(lifetime_value), 2)  AS avg_lifetime_value,
        ROUND(SUM(lifetime_value), 2)  AS total_revenue,
        ROUND(AVG(avg_order_value), 2) AS avg_order_value,
        ROUND(AVG(total_orders), 1)    AS avg_orders_per_customer,
        ROUND(AVG(loyalty_points), 0)  AS avg_loyalty_points,
        ROUND(AVG(annual_income), 2)   AS avg_annual_income
    FROM customer_analytics
""").toPandas()

kpi = df.T.reset_index()
kpi.columns = ["KPI", "Value"]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>KPI</b>", "<b>Value</b>"],
        fill_color="#2E3A59",
        font=dict(color="white", size=13),
        align="left",
        height=35
    ),
    cells=dict(
        values=[kpi["KPI"], kpi["Value"]],
        fill_color=[["#F1EFE8", "#FFFFFF"] * len(kpi)],
        font=dict(size=12),
        align="left",
        height=30
    )
)])
fig.update_layout(title="📊 Customer Analytics — KPI Summary")
fig.show()